# Global Table Intersection Utils 测试 Notebook

这个 notebook 用来测试 `global_table_intersection_utils.py` 的算法是否可用。

核心问题：给定 global detector 输出的 pixel 坐标 `(center_u, center_v)`，如何估计它在桌面上的 3D 坐标？

当前工具使用的是 **RGB pixel ray 与桌面平面求交**，不是 depth deprojection。

也就是说：

1. 从 pixel `(u, v)` 和相机内参 `K` 生成 camera ray。
2. 使用当前机械臂位姿 `T_base_gripper` 和 hand-eye `T_gripper_camera` 得到 camera 在 robot base frame 下的位置和朝向。
3. 假设目标点落在桌面平面 `base_z = z_table` 上。
4. 计算 camera ray 与桌面平面的交点。
5. 同时返回：
   - `center_camera_on_table_m`
   - `center_base_on_table_m`

重要结论：

- 如果你只用 raw aligned depth 反投影到 **camera frame**，只需要 `u, v, depth_m, fx, fy, ppx, ppy`，不需要机械臂当前位姿。
- 如果你用这个 **ray/table intersection** 方法，并且桌面平面定义在 robot base frame，例如 `base_z = 0`，那么需要：
  - camera intrinsics `K`
  - hand-eye `T_gripper_camera`
  - 当前机械臂位姿 `T_base_gripper`
  - table height `z_table`
- 如果要输出 **robot base frame** 坐标，也必须需要当前 `T_base_gripper`。
- 如果没有真实当前机械臂位姿，使用 `np.eye(4)` 只能做代码连通性测试，几何结果没有实际意义。

In [1]:
from pathlib import Path
import json
import math
import copy

import numpy as np
import pandas as pd

from global_table_intersection_utils import (
    RGB_TABLE_INTERSECTION_SOURCE,
    load_hand_eye_npz,
    pixel_to_table_intersection_camera_and_base,
    add_table_intersection_to_detections,
    add_table_intersection_to_result_dict,
)

np.set_printoptions(precision=6, suppress=True)
print("Loaded global_table_intersection_utils.py")
print("source:", RGB_TABLE_INTERSECTION_SOURCE)

Loaded global_table_intersection_utils.py
source: rgb_ray_table_intersection


## 1. 需要哪些输入？

`pixel_to_table_intersection_camera_and_base(...)` 当前需要这些输入：

| 输入 | 含义 | 是否必须 | 说明 |
|---|---|---|---|
| `u, v` | global detection 的 pixel 中心 | 是 | 来自 `center_u`, `center_v` |
| `K` | color camera intrinsics matrix | 是 | `fx, fy, ppx, ppy` 组成的 3x3 matrix |
| `T_gripper_camera` | hand-eye 外参 | 是 | camera 相对于 gripper 的位姿 |
| `T_base_gripper` | 当前机械臂末端在 base 下的位姿 | 是，如果桌面在 base frame 下定义 | 必须是拍摄这张图时的 robot pose |
| `z_table` | 桌面在 base frame 下的高度 | 是 | 常见是 `0.0`，但要看你的 robot/table 标定 |

注意：这个方法没有用 raw depth。它是假设 detected pixel 对应的物体中心投影落在桌面平面上。

In [2]:
def make_K(fx=600.0, fy=600.0, ppx=320.0, ppy=240.0):
    return np.array([
        [fx, 0.0, ppx],
        [0.0, fy, ppy],
        [0.0, 0.0, 1.0],
    ], dtype=np.float64)


def transform_from_R_t(R, t):
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = np.asarray(R, dtype=np.float64)
    T[:3, 3] = np.asarray(t, dtype=np.float64)
    return T


def summarize_result(result):
    keys = [
        "center_px",
        "center_camera_on_table_m",
        "center_base_on_table_m",
        "ray_table_intersection_valid",
        "ray_table_intersection_reason",
        "table_z_m",
        "used_depth_source",
    ]
    return {k: result.get(k) for k in keys}


def assert_close_list(actual, expected, atol=1e-9, name="value"):
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    if not np.allclose(actual, expected, atol=atol):
        raise AssertionError(f"{name} mismatch: actual={actual}, expected={expected}")

print("Helper functions ready")

Helper functions ready


## 2. 合成测试：已知 camera 朝下，桌面在 base_z = 0

这里构造一个最简单、可手算的场景：

- 相机在 base frame 的位置是 `(0, 0, 1)` m。
- camera 的 optical z 轴朝向 base 的负 z 方向，也就是向下看桌面。
- 桌面是 `base_z = 0`。
- 中心 pixel 的 ray 应该打到 base 原点 `(0, 0, 0)`。

这一步的意义是验证算法的几何方向、矩阵乘法和坐标返回是否正确。

In [3]:
K_test = make_K(fx=100.0, fy=100.0, ppx=320.0, ppy=240.0)

# Camera optical z -> base -z. This is a simple synthetic transform for testing.
R_base_camera = np.diag([1.0, 1.0, -1.0])
t_base_camera = np.array([0.0, 0.0, 1.0])
T_base_camera = transform_from_R_t(R_base_camera, t_base_camera)

# The utility computes T_base_camera = T_base_gripper @ T_gripper_camera.
# For this synthetic test, use T_base_gripper=I and put the full camera pose in T_gripper_camera.
T_base_gripper_test = np.eye(4, dtype=np.float64)
T_gripper_camera_test = T_base_camera.copy()
z_table_test = 0.0

center_result = pixel_to_table_intersection_camera_and_base(
    u=320.0,
    v=240.0,
    K=K_test,
    T_base_gripper=T_base_gripper_test,
    T_gripper_camera=T_gripper_camera_test,
    z_table=z_table_test,
)

summarize_result(center_result)

{'center_px': [320.0, 240.0],
 'center_camera_on_table_m': [0.0, 0.0, 1.0],
 'center_base_on_table_m': [0.0, 0.0, 0.0],
 'ray_table_intersection_valid': True,
 'ray_table_intersection_reason': None,
 'table_z_m': 0.0,
 'used_depth_source': 'rgb_ray_table_intersection'}

In [4]:
assert center_result["ray_table_intersection_valid"] is True
assert_close_list(center_result["center_base_on_table_m"], [0.0, 0.0, 0.0], name="center base point")
assert_close_list(center_result["center_camera_on_table_m"], [0.0, 0.0, 1.0], name="center camera point")
print("PASS: center pixel intersects the expected table point")

PASS: center pixel intersects the expected table point


## 3. 合成测试：偏离中心的 pixel

在这个测试里，`fx = fy = 100`，所以 pixel 向右偏移 100 px，对应 normalized camera ray 的 x 分量为 1。

中心为 `(320, 240)`，测试 pixel 为 `(420, 240)`。

预期：

- camera frame 下交点约为 `[1, 0, 1]`
- base frame 下交点约为 `[1, 0, 0]`

In [5]:
off_center_result = pixel_to_table_intersection_camera_and_base(
    u=420.0,
    v=240.0,
    K=K_test,
    T_base_gripper=T_base_gripper_test,
    T_gripper_camera=T_gripper_camera_test,
    z_table=z_table_test,
)

summarize_result(off_center_result)

{'center_px': [420.0, 240.0],
 'center_camera_on_table_m': [1.0, 0.0, 1.0],
 'center_base_on_table_m': [1.0, 0.0, 0.0],
 'ray_table_intersection_valid': True,
 'ray_table_intersection_reason': None,
 'table_z_m': 0.0,
 'used_depth_source': 'rgb_ray_table_intersection'}

In [6]:
assert off_center_result["ray_table_intersection_valid"] is True
assert_close_list(off_center_result["center_base_on_table_m"], [1.0, 0.0, 0.0], name="off-center base point")
assert_close_list(off_center_result["center_camera_on_table_m"], [1.0, 0.0, 1.0], name="off-center camera point")
print("PASS: off-center pixel intersects the expected table point")

PASS: off-center pixel intersects the expected table point


## 4. 失败情况测试

如果相机朝向和桌面几何关系不对，射线可能：

- 与桌面几乎平行；
- 或者交点在 camera 后方；
- 或者输入矩阵/坐标不合法。

工具函数不会 crash，而是返回：

```python
ray_table_intersection_valid = False
ray_table_intersection_reason = "..."
```

In [7]:
# Camera optical z maps to base +z, while camera is already above the table.
# The table intersection will lie behind the camera.
T_bad_camera_pose = transform_from_R_t(np.eye(3), [0.0, 0.0, 1.0])

bad_result = pixel_to_table_intersection_camera_and_base(
    u=320.0,
    v=240.0,
    K=K_test,
    T_base_gripper=np.eye(4),
    T_gripper_camera=T_bad_camera_pose,
    z_table=0.0,
)

summarize_result(bad_result)

{'center_px': [320.0, 240.0],
 'center_camera_on_table_m': None,
 'center_base_on_table_m': None,
 'ray_table_intersection_valid': False,
 'ray_table_intersection_reason': 'table intersection lies behind the camera',
 'table_z_m': None,
 'used_depth_source': 'rgb_ray_table_intersection'}

In [8]:
assert bad_result["ray_table_intersection_valid"] is False
assert bad_result["ray_table_intersection_reason"] is not None
print("PASS: invalid geometry returns a readable failure reason")

PASS: invalid geometry returns a readable failure reason


## 5. 批量给 global detection 结果添加 table intersection 字段

`add_table_intersection_to_detections(...)` 可以直接处理 global detector 输出的 list。

它会保留原来的 detection 字段，同时增加：

- `center_px`
- `center_camera_on_table_m`
- `center_base_on_table_m`
- `ray_table_intersection_valid`
- `ray_table_intersection_reason`
- `table_z_m`
- `used_depth_source`

In [9]:
sample_detections = [
    {"instance_name": "gear_large", "class_name": "gear", "center_u": 320.0, "center_v": 240.0},
    {"instance_name": "gear_small", "class_name": "gear", "center_u": 420.0, "center_v": 240.0},
    {"instance_name": "base_hole_m", "class_name": "base_hole", "center_u": 320.0, "center_v": 340.0},
]

augmented = add_table_intersection_to_detections(
    sample_detections,
    K=K_test,
    T_base_gripper=T_base_gripper_test,
    T_gripper_camera=T_gripper_camera_test,
    z_table=z_table_test,
)

pd.DataFrame(augmented)

,instance_name,class_name,center_u,center_v,center_px,camera_x_m,camera_y_m,camera_z_m,camera_frame_id,depth_valid,depth_m,depth_source,ray_table_intersection_valid,ray_table_intersection_reason,table_z_m,center_camera_on_table_m,used_depth_source
0,gear_large,gear,320.0,240.0,"[320.0, 240.0]",0.0,0.0,1.0,camera_color_optical_frame,True,1.0,rgb_ray_table_intersection,True,None,0.0,"[0.0, 0.0, 1.0]",rgb_ray_table_intersection
1,gear_small,gear,420.0,240.0,"[420.0, 240.0]",1.0,0.0,1.0,camera_color_optical_frame,True,1.0,rgb_ray_table_intersection,True,None,0.0,"[1.0, 0.0, 1.0]",rgb_ray_table_intersection
2,base_hole_m,base_hole,320.0,340.0,"[320.0, 340.0]",0.0,1.0,1.0,camera_color_optical_frame,True,1.0,rgb_ray_table_intersection,True,None,0.0,"[0.0, 1.0, 1.0]",rgb_ray_table_intersection


## 6. 真实 hand-eye 文件检查

下面会尝试读取当前项目里的 hand-eye 结果：

```text
cam_calibration/outputs/03_hand_eye/hand_eye_result.npz
```

注意：hand-eye 只提供 `T_gripper_camera` 和 camera intrinsics。

它不提供拍摄当前 global 图像时的 `T_base_gripper`。这个位姿必须来自机器人端，例如 Franka 当前 EE pose。

In [10]:
HAND_EYE_NPZ_PATH = Path("cam_calibration/outputs/03_hand_eye/hand_eye_result.npz")

if HAND_EYE_NPZ_PATH.exists():
    T_gripper_camera_npz, CAMERA_MATRIX_NPZ, DIST_COEFFS_NPZ = load_hand_eye_npz(HAND_EYE_NPZ_PATH)
    print("Loaded:", HAND_EYE_NPZ_PATH)
    print("camera_matrix K:\n", CAMERA_MATRIX_NPZ)
    print("T_gripper_camera:\n", T_gripper_camera_npz)
    print("dist_coeffs shape:", DIST_COEFFS_NPZ.shape)
else:
    T_gripper_camera_npz = None
    CAMERA_MATRIX_NPZ = None
    DIST_COEFFS_NPZ = None
    print("Missing hand-eye file:", HAND_EYE_NPZ_PATH)

Loaded: cam_calibration/outputs/03_hand_eye/hand_eye_result.npz
camera_matrix K:
 [[605.374512   0.       318.154175]
 [  0.       604.334045 255.365128]
 [  0.         0.         1.      ]]
T_gripper_camera:
 [[-0.00142   0.999943 -0.010601 -0.051506]
 [-0.999984 -0.001478 -0.005485  0.02972 ]
 [-0.0055    0.010593  0.999929  0.059968]
 [ 0.        0.        0.        1.      ]]
dist_coeffs shape: (5,)


## 7. 使用真实 hand-eye + 手动 robot pose 测试

这里是你后续真正需要填的地方。

请把拍摄 global RGB 图片时的 Franka 当前末端位姿填到 `T_base_gripper_manual`。

如果这里继续使用 `np.eye(4)`，测试只能说明代码能跑，不能说明几何结果可用。

In [11]:
# Replace this matrix with the real Franka T_base_gripper at the moment of image capture.
# Units: meter. Rotation must be a valid 3x3 rotation matrix.
T_base_gripper_manual = np.eye(4, dtype=np.float64)

# Set this to your table height in robot base frame.
TABLE_Z_M = 0.0

print("T_base_gripper_manual currently is identity:", np.allclose(T_base_gripper_manual, np.eye(4)))
print("TABLE_Z_M:", TABLE_Z_M)

T_base_gripper_manual currently is identity: True
TABLE_Z_M: 0.0


In [12]:
if CAMERA_MATRIX_NPZ is not None:
    test_pixel_u = 320.0
    test_pixel_v = 240.0
    real_like_result = pixel_to_table_intersection_camera_and_base(
        u=test_pixel_u,
        v=test_pixel_v,
        K=CAMERA_MATRIX_NPZ,
        T_base_gripper=T_base_gripper_manual,
        T_gripper_camera=T_gripper_camera_npz,
        z_table=TABLE_Z_M,
    )
    display(pd.DataFrame([summarize_result(real_like_result)]))
else:
    print("Skip: hand-eye npz was not loaded")

,center_px,center_camera_on_table_m,center_base_on_table_m,ray_table_intersection_valid,ray_table_intersection_reason,table_z_m,used_depth_source
0,"[320.0, 240.0]",None,None,False,table intersection lies behind the camera,None,rgb_ray_table_intersection


## 8. 读取已有 global detection JSON 并尝试增强

这里会尝试读取：

```text
output/global_detection/global_detection_results.json
```

如果文件存在，会把每个 detection 的 `center_u / center_v` 通过 table intersection 增强。

再次提醒：如果 `T_base_gripper_manual` 还是 identity，输出坐标不能用于真实机器人。

In [13]:
GLOBAL_JSON_PATH = Path("output/global_detection/global_detection_results.json")

if GLOBAL_JSON_PATH.exists() and CAMERA_MATRIX_NPZ is not None:
    global_data = json.loads(GLOBAL_JSON_PATH.read_text())
    print("Loaded:", GLOBAL_JSON_PATH)

    if isinstance(global_data, list):
        detections = global_data
        augmented_global = add_table_intersection_to_detections(
            detections,
            K=CAMERA_MATRIX_NPZ,
            T_base_gripper=T_base_gripper_manual,
            T_gripper_camera=T_gripper_camera_npz,
            z_table=TABLE_Z_M,
        )
    elif isinstance(global_data, dict):
        augmented_dict = add_table_intersection_to_result_dict(
            global_data,
            K=CAMERA_MATRIX_NPZ,
            T_base_gripper=T_base_gripper_manual,
            T_gripper_camera=T_gripper_camera_npz,
            z_table=TABLE_Z_M,
        )
        augmented_global = augmented_dict.get("detections", [])
    else:
        augmented_global = []
        print("Unexpected JSON structure:", type(global_data))

    columns = [
        "instance_name",
        "class_name",
        "center_u",
        "center_v",
        "center_camera_on_table_m",
        "center_base_on_table_m",
        "ray_table_intersection_valid",
        "ray_table_intersection_reason",
    ]
    display(pd.DataFrame([{k: item.get(k) for k in columns} for item in augmented_global]))
else:
    print("Skip. Missing global JSON or hand-eye npz.")
    print("GLOBAL_JSON_PATH exists:", GLOBAL_JSON_PATH.exists())
    print("CAMERA_MATRIX_NPZ loaded:", CAMERA_MATRIX_NPZ is not None)

Loaded: output/global_detection/global_detection_results.json


,instance_name,class_name,center_u,center_v,center_camera_on_table_m,center_base_on_table_m,ray_table_intersection_valid,ray_table_intersection_reason
0,None,Gear_compound,217.09,37.93,None,None,False,table intersection lies behind the camera


## 9. 敏感性测试：机械臂位姿对结果的影响

这个测试回答一个关键问题：

> 是不是需要机械臂目前的位姿？

如果桌面平面定义在 base frame 下，答案是：**需要**。

原因是同一个 pixel ray，从不同 camera pose 出发，会打到桌面的不同位置。

下面故意改变 `T_base_gripper` 的平移，看输出如何变化。

In [14]:
pose_rows = []
for x_shift in [-0.10, 0.0, 0.10]:
    T_bg = np.eye(4, dtype=np.float64)
    T_bg[:3, 3] = [x_shift, 0.0, 0.0]
    r = pixel_to_table_intersection_camera_and_base(
        u=320.0,
        v=240.0,
        K=K_test,
        T_base_gripper=T_bg,
        T_gripper_camera=T_gripper_camera_test,
        z_table=0.0,
    )
    pose_rows.append({
        "x_shift_in_T_base_gripper_m": x_shift,
        "valid": r["ray_table_intersection_valid"],
        "camera_point": r["center_camera_on_table_m"],
        "base_point": r["center_base_on_table_m"],
        "reason": r["ray_table_intersection_reason"],
    })

pd.DataFrame(pose_rows)

,x_shift_in_T_base_gripper_m,valid,camera_point,base_point,reason
0,-0.1,True,"[0.0, 0.0, 1.0]","[-0.1, 0.0, 0.0]",None
1,0.0,True,"[0.0, 0.0, 1.0]","[0.0, 0.0, 0.0]",None
2,0.1,True,"[0.0, 0.0, 1.0]","[0.1, 0.0, 0.0]",None


## 10. 和 raw depth 反投影的区别

如果你有 aligned raw depth，camera frame 坐标可以直接算：

```text
X = (u - ppx) / fx * depth_m
Y = (v - ppy) / fy * depth_m
Z = depth_m
```

这个方法不需要 `T_base_gripper`，也不需要 hand-eye。

但是它需要该 pixel 或周围窗口有可靠 raw depth。

对比：

| 方法 | 需要 depth | 需要当前机械臂位姿 | 输出 camera frame | 输出 base frame | 适合场景 |
|---|---:|---:|---:|---:|---|
| raw depth deprojection | 是 | 否 | 是 | 需要额外 TF/hand-eye/pose | 精确 3D，局部抓取 |
| ray/table intersection | 否 | 是，如果 table 在 base 下 | 是 | 是 | global 粗定位、桌面上目标估计 |

对你的 global 阶段来说，ray/table intersection 可以作为“没有可靠 depth 时的粗略 3D”。
对精确 grasp point，还是建议用 local RGB-D refinement。

In [15]:
def deproject_pixel_with_depth(u, v, depth_m, K):
    K = np.asarray(K, dtype=np.float64)
    fx = K[0, 0]
    fy = K[1, 1]
    ppx = K[0, 2]
    ppy = K[1, 2]
    x = (float(u) - ppx) / fx * float(depth_m)
    y = (float(v) - ppy) / fy * float(depth_m)
    z = float(depth_m)
    return [float(x), float(y), float(z)]

print("Example raw-depth deprojection:")
print(deproject_pixel_with_depth(420.0, 240.0, 1.0, K_test))
print("This matches the synthetic table-intersection camera point only because depth is exactly 1.0 m in the synthetic setup.")

Example raw-depth deprojection:
[1.0, 0.0, 1.0]
This matches the synthetic table-intersection camera point only because depth is exactly 1.0 m in the synthetic setup.


## 11. Checklist：判断这个算法结果是否可用

建议按下面顺序检查：

1. 合成测试必须全部 PASS。
2. hand-eye npz 能正常读取，`T_gripper_camera` 和 `camera_matrix` 数值合理。
3. `T_base_gripper_manual` 必须换成拍摄当前图像时的真实 Franka pose。
4. `z_table` 必须是 robot base frame 下真实桌面高度。
5. 对几个已知位置的桌面点做人工验证，例如把检测点打到 base frame 后，看它是否落在真实桌面对应位置。
6. 如果 camera 在拍摄时移动，必须使用同一时刻的 robot pose，不要拿错时间戳。
7. 对于物体顶部抓取点，不要用 table intersection 当最终精确 3D；它只是假设点在桌面上。

最终建议：

- global 阶段：可以用这个方法给所有 detected part 一个桌面粗定位。
- robot 选择目标后靠近。
- local 阶段：用 raw aligned depth 获取真正 grasp point 的 camera frame 坐标。